In [1]:
from scripts.utils import *
from scripts.setup_evcouplings import *
from scripts.get_structures_and_sequences import *
from scripts.structural_filter import *
from scripts.hmm_domain_filter import *
import time 

In [ ]:
# IMPORTANT PARAMETERS TO SET BEFORE RUNNING:
# evcouplings, foldseek, USalign, hmmpress and hmmsearch should be installed and in Path

SEED_uid = "Q5SK37"
SEED_region_start = 507
SEED_region_end = 569
SEED = [SEED_uid, SEED_region_start, SEED_region_end]
BIT_SCORES = [0.8]
TM_SCORE_CUTOFF = 0.6
HMMs_DIR = "STAC_hmm_domains"
CPUS = 16

CONFIG_TEMPLATE = "evcouplings_config_template.yaml"
number_tracker = {} # dictionary to keep track of true members at each stage in the pipeline

In [3]:
# directory setup for each run:

# root_dir
    # data
        # evcouplings
            # contains config file and output
        # filtering
            # usalgin_dir
            # foldseek_dir
            # structural_filtering_output.csv
            # True_Members.csv
            # structures_containing_seeds_and_not_al_domains.csv
            # hmm_domain_filtering_dir
        # alphafold_structures
            # contains structures of potential true members
        # seed
            # contains structure of seed
        # HMMs
            # contains HMMs
        # all_sequences_full.fasta
        # results
            # contains final results of pipeline to easily copy them


def setup_directories():
    root_dir = f"runs/true_members_run_{SEED[0]}_{time.strftime('%Y-%m-%d_%H-%M-%S')}"
    directories = [
        f"{root_dir}/data/evcouplings",
        f"{root_dir}/data/filtering/usalign",
        f"{root_dir}/data/filtering/foldseek",
        f"{root_dir}/data/filtering/hmm_domain_filtering",
        f"{root_dir}/data/alphafold_structures",
        f"{root_dir}/data/seed",
        f"{root_dir}/data/HMMs",
        f"{root_dir}/data/results"
    ]
    for directory in directories:
        # clear directory if it already exists
        if os.path.exists(directory):
            shutil.rmtree(directory)
        os.makedirs(directory, exist_ok=True)
    return root_dir
        
root_dir = setup_directories()

In [4]:
# Run evcouplings align and couplings stage
run_evcouplings(SEED, BIT_SCORES, CPUS, CONFIG_TEMPLATE, root_dir)

FileNotFoundError: [Errno 2] No such file or directory: 'evcouplings'

In [5]:
# Get all structures and full fasta of all sequences of evcouplings output
EVCOUPLINGS_OUT_DIR = f"{root_dir}/data/evcouplings",
number_tracker = get_all_structures_and_sequences(SEED, BIT_SCORES, number_tracker, root_dir)

Could not extract UniProt accession from record ID: UniRef90_UPI000542879D/506-563
Could not extract UniProt accession from record ID: UniRef90_UPI001FAAB0B1/508-565
Could not extract UniProt accession from record ID: UniRef90_UPI0003824F5A/507-565
Could not extract UniProt accession from record ID: UniRef90_UPI00023898AB/505-563
Could not extract UniProt accession from record ID: UniRef90_UPI000DF2BD3B/508-565
Could not extract UniProt accession from record ID: UniRef90_UPI00263A1D77/508-565
Total unique uniprot UIDs for AlphaFold: 13/19

Skipping entry 'A0ABV6Q133': could not download AlphaFold structure.
Processing 2/13 structures...
Processing 4/13 structures...
Processing 6/13 structures...
Processing 8/13 structures...
Processing 10/13 structures...
Processing 12/13 structures...
Processing 13/13 structures...
Downloaded 12/13 structure files (1 failed).
Fetched 12/12 records ...


In [ ]:
# Run USalgin, foldseek and filter output
number_tracker = run_structural_similarity_filter(SEED, number_tracker, TM_SCORE_CUTOFF, root_dir)

Processing 1/1 structures...
Downloaded 1/1 structure files (0 failed).
Extracting region (507, 569) from protein file: runs/true_members_run_Q5SK37_2026-06-05_11-55-48/data/seed/Q5SK37.cif
Remaining structures after filtering of foldseek output: 11/11
Remaining structures after filtering usalign output: 11/12


In [7]:
# Run hmmsearch with CbrA related HMMs on fasta of STAC containing sequences and filter to get STAC containing CbrAs and STAC containing proteins without CbrA domains
seed_containing_sequences_ids_path = f"{root_dir}/data/filtering/structural_filtering_output_tm_0.6.csv"
number_tracker = run_hmm_domain_filter(number_tracker, CPUS, seed_containing_sequences_ids_path, HMMs_DIR, root_dir)

Found 11/11 matching records in the FASTA file


In [9]:
# Print number of true members at each stage in the pipeline and save results in results directory
for key, value in number_tracker.items():
    print(f"{key}: {value}")
    
df_number_tracker = pd.DataFrame(list(number_tracker.items()), columns=["Stage", "Numbers"])
df_number_tracker.to_csv(f"{root_dir}/data/results/number_tracker.csv", index=False)

copy_file(f"{root_dir}/data/filtering/True_Members.csv", f"{root_dir}/data/results/True_Members.csv")
copy_file(f"{root_dir}/data/filtering/structures_with_seed_and_not_all_domains.csv", f"{root_dir}/data/results/structures_with_seed_and_not_all_domains.csv")
    

annotation_files: 1
unique_ids: 19
unique_uniprot_uids: 13
successful_structure_downloads: 12
downloaded_records: 12
number_filtered_structures_foldseek: 11
number_filtered_structures_usalign: 11
number_filtered_structures_total: 11
structures_with_seed_and_domains: 0
structures_with_seed_and_not_all_domains: 11


True

In [ ]:
# Feed results into STAC containing CbrA and STAC containing protein datasets and compute statistics 